# All About LLMs: Positional Encoding & RoPE

This notebook builds up the theory of positional encoding in Transformers from first principles, then contrasts it with the Rotary Positional Embeddings (RoPE) used in modern LLMs (Llama, Mistral, Gemma, Qwen, Phi, etc.).

**Contents**
1. Why Transformers need positional information at all
2. Sinusoidal positional encoding — formula and design rationale
3. Key mathematical properties (relative position, uniqueness, boundedness, extrapolation)
4. Worked numerical example
5. RoPE — motivation and formula
6. Worked numerical example for RoPE
7. Comparison and summary

## 1. Why Positional Encoding Is Needed

Self-attention is **permutation-invariant**: it computes weighted sums over all tokens regardless of their order. Without extra information, "the dog bit the man" and "the man bit the dog" would look identical to the attention mechanism, because the set of token embeddings is the same. Transformers therefore inject explicit position information into the input.

The original Transformer ("Attention Is All You Need") does this with a fixed **sinusoidal positional encoding** added to the token embeddings before the first layer.

## 2. Sinusoidal Positional Encoding — Formula

For position $pos$ and dimension index $i$ (with $d_{model}$ total embedding dimensions):

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Each pair of dimensions $(2i, 2i+1)$ oscillates at its own frequency, and the final input to the model is:

$$\text{Final Input}[pos] = \text{TokenEmbedding}[pos] + PE[pos]$$

### Why these frequencies?

The term $10000^{2i/d_{model}}$ controls the wavelength for each dimension pair:

- Small $i$ (low dimensions) → divisor ≈ 1 → **high frequency**, fast oscillation.
- Large $i$ (high dimensions) → divisor is large → **low frequency**, slow oscillation.

This produces a geometric progression of wavelengths, from about $2\pi$ up to $10000 \times 2\pi$. Different dimensions "tick" at different rates, giving the model positional signal at multiple scales simultaneously — fast pairs distinguish nearby positions, slow pairs distinguish coarse, long-range position.

## 3. Key Mathematical Properties

**a) Relative position via a linear (rotation) relationship — the most important property.**
Using the angle-addition identities
$$\sin(a+b) = \sin a\cos b + \cos a \sin b, \qquad \cos(a+b) = \cos a \cos b - \sin a \sin b$$
each 2D pair $(PE(pos,2i), PE(pos,2i+1))$ shifted by a fixed offset $k$ is exactly a **rotation**:

$$\begin{bmatrix} PE(pos+k, 2i) \\ PE(pos+k, 2i+1) \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} PE(pos, 2i) \\ PE(pos, 2i+1) \end{bmatrix}, \qquad \theta = \frac{k}{10000^{2i/d_{model}}}$$

Because this is a *linear* transformation, attention (which is built from linear projections and dot products) can learn to recover relative distance $k$ between tokens without needing to learn a complex nonlinear function.

**b) Periodicity + uniqueness.** Sine and cosine are individually periodic, but because every dimension pair oscillates at a different frequency, the combination of values across the full $d_{model}$-length vector is unique for each position within a very long range.

**c) Bounded values.** Every entry stays in $[-1, 1]$, matching the scale of normalized embeddings and helping training stability.

**d) Extrapolation.** Because $PE$ is a fixed closed-form function (not learned), it can be evaluated at positions longer than any sequence seen in training — unlike learned positional embedding tables, which have no representation for out-of-range positions.

## 4. Worked Numerical Example (Sinusoidal PE)

**Setup:** `d_model = 4` → 2 dimension pairs.
- Pair 1 ($i=0$): columns 0 (sin), 1 (cos) — divisor $10000^{0/4} = 1$, so $\theta_0(pos) = pos/1 = pos$ (**fast** frequency).
- Pair 2 ($i=1$): columns 2 (sin), 3 (cos) — divisor $10000^{2/4} = 100$, so $\theta_1(pos) = pos/100$ (**slow** frequency).

**Sentence:** "I like playing football in my leisure time"

### Step 1 — compute the angles $\theta_0$ and $\theta_1$ for each position

| Word | pos | $\theta_0 = pos/1$ | $\theta_1 = pos/100$ |
|------|-----|---------------------|------------------------|
| I | 0 | 0.00 | 0.0000 |
| like | 1 | 1.00 | 0.0100 |
| playing | 2 | 2.00 | 0.0200 |
| football | 3 | 3.00 | 0.0300 |
| in | 4 | 4.00 | 0.0400 |
| my | 5 | 5.00 | 0.0500 |
| leisure | 6 | 6.00 | 0.0600 |
| time | 7 | 7.00 | 0.0700 |

### Step 2 — take sin/cos of each angle to get the PE vector

| Word | pos | Col0 sin($\theta_0$) | Col1 cos($\theta_0$) | Col2 sin($\theta_1$) | Col3 cos($\theta_1$) |
|------|-----|------------------------|------------------------|------------------------|------------------------|
| I | 0 | 0.0000 | 1.0000 | 0.0000 | 1.0000 |
| like | 1 | 0.8415 | 0.5403 | 0.0100 | 0.99995 |
| playing | 2 | 0.9093 | -0.4161 | 0.0200 | 0.99980 |
| football | 3 | 0.1411 | -0.9900 | 0.0300 | 0.99955 |
| in | 4 | -0.7568 | -0.6536 | 0.0400 | 0.99920 |
| my | 5 | -0.9589 | 0.2837 | 0.0500 | 0.99875 |
| leisure | 6 | -0.2794 | 0.9602 | 0.0600 | 0.99820 |
| time | 7 | 0.6570 | 0.7539 | 0.0699 | 0.99755 |

### Adding to token embeddings (dummy example)

| Word | Dummy Token Embedding | Positional Encoding | **Final Input** |
|------|------------------------|----------------------|-------------------|
| I | [0.20, -0.50, 0.40, 0.10] | [0.0000, 1.0000, 0.0000, 1.0000] | [0.2000, 0.5000, 0.4000, 1.1000] |
| like | [0.10, 0.60, -0.30, 0.80] | [0.8415, 0.5403, 0.0100, 0.99995] | [0.9415, 1.1403, -0.2900, 1.79995] |
| playing | [0.50, -0.30, 0.80, 0.20] | [0.9093, -0.4161, 0.0200, 0.99980] | [1.4093, -0.7161, 0.8200, 1.19980] |
| football | [0.30, 0.40, -0.20, 0.60] | [0.1411, -0.9900, 0.0300, 0.99955] | [0.4411, -0.5900, -0.1700, 1.59955] |

### Verifying the rotation property

Take "like" ($pos=1$) → "football" ($pos=3$), so $k=2$. For the fast pair ($i=0$), the shift angle is

$$\theta = \frac{k}{10000^{2i/d_{model}}} = \frac{2}{10000^{0/4}} = \frac{2}{1} = 2 \text{ radians}$$

and the pair is stored in $(x,y) = (\cos(pos), \sin(pos))$ order. Shifting the angle by $\theta=2$ radians should reproduce $(\cos(3), \sin(3)) = (-0.9900, 0.1411)$. Starting from $(\cos(1), \sin(1)) = (0.5403, 0.8415)$ and applying the rotation matrix:

$$\begin{bmatrix}x'\\y'\end{bmatrix} = \begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix}\begin{bmatrix}x\\y\end{bmatrix}, \qquad \cos(2)\approx-0.4161,\ \sin(2)\approx0.9093$$

$$x' = \cos(2)(0.5403) - \sin(2)(0.8415) = (-0.4161)(0.5403) - (0.9093)(0.8415) = -0.2248 - 0.7652 = -0.9900 = \cos(3)\ \checkmark$$
$$y' = \sin(2)(0.5403) + \cos(2)(0.8415) = (0.9093)(0.5403) + (-0.4161)(0.8415) = 0.4913 - 0.3501 = 0.1412 \approx \sin(3)\ \checkmark$$

For the slow pair ($i=1$), the same offset $k=2$ gives a much smaller shift angle:

$$\theta = \frac{k}{10000^{2i/d_{model}}} = \frac{2}{10000^{2/4}} = \frac{2}{100} = 0.02 \text{ radians}$$

confirming that low dimensions rotate fast per step while high dimensions barely rotate — the geometric progression of frequencies described in Section 2.

The rotation exactly reproduces $PE_{(3,\cdot)}$ from $PE_{(1,\cdot)}$, confirming that a fixed offset $k$ corresponds to a fixed rotation angle, independent of the starting position.

## 5. Dot Products Between Positional Encodings

Attention scores involve dot products of (embedding + PE) vectors. The PE component of that dot product depends only on relative distance $k$, which is what lets the model use position information without extra learned parameters.

| From → To | k | Dot product of PE vectors | Interpretation |
|-----------|---|---------------------------|-----------------|
| I → I | 0 | 2.0000 | maximum (identical position) |
| I → like | 1 | 1.5403 | high similarity |
| I → playing | 2 | 0.5837 | moderate |
| like → leisure | 5 | 1.2825 | still fairly high — periodicity means similarity doesn't decay monotonically with distance |

Example calculation (I → like, $k=1$): $PE_0=[0,1,0,1]$, $PE_1=[0.8415, 0.5403, 0.0100, 0.99995]$

$$\text{dot} = 0\times0.8415 + 1\times0.5403 + 0\times0.0100 + 1\times0.99995 = 1.54025$$

**Limitation to note:** because sin/cos are periodic, the dot product is not a monotonic function of $k$ — very distant positions can occasionally look similar again. This non-monotonic decay, plus the fact that positional information is only *added* (and can be partially washed out by deep stacks of layers), is part of the motivation for RoPE, covered next.

## 6. RoPE — Rotary Positional Embeddings

### Quick refresher: radians

A radian is the unit of rotation angle on the unit circle (radius = 1). $\text{degrees} = \text{radians}\times\frac{180}{\pi}$. Landmarks: $0$ rad → $(1,0)$; $\pi/2\approx1.57$ rad → $(0,1)$ (90°); $\pi\approx3.14$ rad → $(-1,0)$ (180°); $2\pi\approx6.28$ rad → back to $(1,0)$ (360°).

### Motivation

Sinusoidal PE **adds** a position vector to the token embedding once, at the input layer. RoPE — used in Llama, Mistral, Gemma, Qwen, Phi, and most modern LLMs — takes a different approach: instead of adding anything, it **rotates** the Query and Key vectors inside every attention layer by an angle that depends on token position.

Advantages over additive sinusoidal PE:
- Relative position is encoded **explicitly** in the attention dot product, not just recoverable in principle.
- Better extrapolation to sequence lengths beyond training.
- No positional signal is mixed into the value/residual stream — it lives purely in how Q and K interact.

### Formula

For dimension-pair index $m$ and position $pos$, with $d_{model}$ total dimensions:

$$\theta_m(pos) = \frac{pos}{10000^{2m/d_{model}}}$$

For a vector $x = [x_0, x_1, x_2, x_3, \dots]$, grouped into pairs $(x_{2m}, x_{2m+1})$, RoPE rotates each pair:

$$\begin{bmatrix} x'_{2m} \\ x'_{2m+1} \end{bmatrix} = \begin{bmatrix} \cos\theta_m & -\sin\theta_m \\ \sin\theta_m & \cos\theta_m \end{bmatrix} \begin{bmatrix} x_{2m} \\ x_{2m+1} \end{bmatrix}$$

This rotation is applied to the **Query** and **Key** projections (not the Value, and not the raw token embedding) inside every attention layer.

### Why the attention score depends only on relative position

The attention score between a rotated query at position $i$ and a rotated key at position $j$ is:

$$q_i^\top k_j = \left(R(\theta_i)\, W_q x_i\right)^\top \left(R(\theta_j)\, W_k x_j\right) = (W_q x_i)^\top R(\theta_i)^\top R(\theta_j) (W_k x_j)$$

Because rotation matrices compose ($R(\theta_i)^\top R(\theta_j) = R(\theta_j - \theta_i)$), this depends only on $\theta_j - \theta_i$, i.e. only on the **relative distance** $k = j - i$ — never on the absolute positions $i$ or $j$ individually. This is a strictly stronger relative-position guarantee than sinusoidal PE provides.

## 7. Worked Numerical Example (RoPE)

**Setup:** same `d_model = 4` toy embeddings as before, applied to the Query/Key vector for each word (here just illustrated on the raw dummy embedding, standing in for a Query or Key vector).

- Fast pair ($m=0$): $\theta_0(pos) = pos$
- Slow pair ($m=1$): $\theta_1(pos) = pos/100$

**Rotating "football"'s vector (pos = 3), dummy vector $[0.30, 0.40, -0.20, 0.60]$**

Fast pair, $\theta_0 = 3$ rad, $\cos(3)\approx-0.9900$, $\sin(3)\approx0.1411$:

$$x'_0 = \cos(3)(0.30) - \sin(3)(0.40) = -0.2970 - 0.0564 = -0.3534$$
$$x'_1 = \sin(3)(0.30) + \cos(3)(0.40) = 0.0423 - 0.3960 = -0.3537$$

Slow pair, $\theta_1 = 0.03$ rad, $\cos(0.03)\approx0.99955$, $\sin(0.03)\approx0.030$:

$$x'_2 = \cos(0.03)(-0.20) - \sin(0.03)(0.60) = -0.19991 - 0.0180 = -0.2179$$
$$x'_3 = \sin(0.03)(-0.20) + \cos(0.03)(0.60) = -0.0060 + 0.5997 = 0.5937$$

**Rotated vector:** $[-0.3534, -0.3537, -0.2179, 0.5937]$

### Relative-distance check: "like" (pos=1) → "football" (pos=3)

Fast-pair relative angle: $\theta_0(3) - \theta_0(1) = 3 - 1 = 2$ rad. Slow-pair relative angle: $\theta_1(3)-\theta_1(1) = 0.03-0.01 = 0.02$ rad. Exactly the same offsets used for the sinusoidal-PE rotation example in section 4 — because both schemes use the same $10000^{2i/d_{model}}$ frequency base, only *where* the rotation is applied differs (added once to the embedding vs. applied to Q/K every layer).

## 8. Comparison: Sinusoidal PE vs. RoPE

| Aspect | Sinusoidal PE | RoPE |
|--------|----------------|------|
| Where applied | Added to token embedding, once, at input | Applied to Query & Key, inside every attention layer |
| Relative position | Indirect — recoverable via dot product | Direct — provably depends only on $i-j$ |
| Length extrapolation | Moderate | Strong |
| Learned parameters | None (fixed function) | None (fixed function) |
| Used in | Original Transformer, BERT | Llama, Mistral, Gemma, Qwen, Phi, most current LLMs |

## 9. Summary

- Self-attention has no built-in notion of order, so position must be injected explicitly.
- **Sinusoidal PE** adds a fixed sin/cos vector to each token embedding; its frequencies form a geometric progression across dimensions, and a positional shift by $k$ corresponds exactly to a 2D rotation of each dimension pair — which is what lets attention recover relative distance.
- **RoPE** moves the same rotation idea into the attention mechanism itself, rotating Query and Key vectors directly so the attention score is a function of relative position only. This is why RoPE has become the default choice in modern LLMs — cleaner relative-position signal and markedly better extrapolation to longer contexts.